In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, List, Annotated
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import operator
import os

In [ ]:
load_dotenv()
GOOGLE_APT_KEY = os.getenv("GOOGLE_API_KEY")

In [ ]:
model = ChatGoogleGenerativeAI(
    api_key = GOOGLE_APT_KEY,
    model = "gemini-2.5-flash-lite"
)

In [ ]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(descrpition= "Detailed feedback for the essay.")
    score: int = Field(descrpition= "Score out of 10.", ge=0, le=10)

In [ ]:
structure_model = model.with_structured_output(EvaluationSchema)

In [ ]:
essay = """Freelancing offers a modern path to professional autonomy, allowing individuals to operate as independent contractors rather than traditional employees. Enabled by digital platforms, freelancers trade their skills—ranging from software development to creative writing—directly with global clients.

This model grants immense freedom, giving professionals control over their workspace, project selection, and daily schedules. However, this flexibility demands strict self-discipline. Freelancers must independently navigate unpredictable income streams, manage their own administrative tasks, and constantly seek new opportunities. Ultimately, freelancing shifts the traditional employment paradigm, transforming career growth into a self-directed journey defined by personal initiative and adaptability."""

In [ ]:
prompt = f"Evaluate the language quality of the following essay and provide a feedbase and assign a score out of 10 \n {essay}"
structure_model.invoke(prompt).score

In [ ]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clartiy_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add] # reducer functions 
    avg_score: float

In [ ]:
def evaluate_language(state: UPSCState):
    try:
        prompt = f"Evaluate the language quality of the following essay and provide a feedbase and assign a score out of 10 \n {state['essay']}"
        output = structure_model.invoke(prompt)
        return{
            "language_feedback":output.feedback,
            "individual_scores": [output.score]
        }
    except Exception as e:
        print(f"Error in chat_node: {e}")
        raise

In [ ]:
def evaluate_anaylsis(state: UPSCState):
    try:
        prompt = f"Evaluate the depth of anaylsis of the following essay and provide a feedbase and assign a score out of 10 \n {state['essay']}"
        output = structure_model.invoke(prompt)
        return{
            "analysis_feedback":output.feedback,
            "individual_scores": [output.score]
        }
    except Exception as e:
        print(f"Error in chat_node: {e}")
        raise

In [ ]:
def evaluate_thought(state: UPSCState):
    try:
        prompt = f"Evaluate the clartiy of thought of the following essay and provide a feedbase and assign a score out of 10 \n {state['essay']}"
        output = structure_model.invoke(prompt)
        return{
            "clartiy_feedback":output.feedback,
            "individual_scores": [output.score]
        }
    except Exception as e:
        print(f"Error in chat_node: {e}")
        raise

In [ ]:
def final_evaluate(state: UPSCState):
    try:
        prompt = f"""
            Based on the following feedback components, create a single, cohesive, summarized feedback paragraph.

            1. Language Feedback:
            {state['language_feedback']}

            2. Depth of Analysis Feedback:
            {state['analysis_feedback']}

            3. Clarity of Thought Feedback:
            {state['clartiy_feedback']}
            """
        overall_feedback = model.invoke(prompt).content

        avg_score= sum(state['individual_scores'])/len(state['individual_scores'])

        return{
            "overall_feedback":overall_feedback ,
            "avg_score":avg_score
        }
    except Exception as e:
        print(f"Error in chat_node: {e}")
        raise


In [ ]:
graph = StateGraph(UPSCState)

graph.add_node("evaluate_language",evaluate_language)
graph.add_node("evaluate_anaylsis",evaluate_anaylsis)
graph.add_node("evaluate_thought",evaluate_thought)
graph.add_node("final_evaluate",final_evaluate)

graph.add_edge(START,"evaluate_language")
graph.add_edge(START,"evaluate_anaylsis")
graph.add_edge(START,"evaluate_thought")

graph.add_edge("evaluate_language","final_evaluate")
graph.add_edge("evaluate_anaylsis","final_evaluate")
graph.add_edge("evaluate_thought","final_evaluate")

graph.add_edge("final_evaluate", END)

workflow = graph.compile()

In [ ]:
workflow

In [ ]:
initial_state = {
    'essay': essay
}
final_state = workflow.invoke(initial_state)

In [ ]:
overall_feedback = final_state ["overall_feedback"]

print(answer.content[0]["text"])

print(answer.content[2]["individual_scores"])

print(answer.content[3]["avg_score"])